In [ ]:
"""
Enhanced Gene-to-m/z Spatial Pattern Matching Pipeline V3
=========================================================

KEY IMPROVEMENTS IN V3:
1. Proper neighbor structures (6 for hexagonal RNA, 8 for Cartesian MSI)
2. Artifact/outlier detection and filtering (blood spots, extreme values)
3. Spatial coherence scoring to avoid isolated bright spots
4. Multi-scale pattern matching
5. Cross-sample analysis across ALL groups (YC, YAD, AC, AAD)
6. Better importance scoring that penalizes isolated hotspots

Author: Enhanced version V3
"""

import numpy as np
import scanpy as sc
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GATConv
from torch_geometric.utils import softmax, add_self_loops
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.neighbors import NearestNeighbors
from scipy.spatial import distance_matrix
from scipy.stats import pearsonr, spearmanr, zscore
from scipy.interpolate import griddata
from scipy.ndimage import gaussian_filter, median_filter
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import seaborn as sns
from typing import List, Dict, Tuple, Optional
import pandas as pd
import os
import warnings
from collections import defaultdict
from dataclasses import dataclass

warnings.filterwarnings('ignore')


# =============================================================================
# DATA CONFIGURATION
# =============================================================================

MSI_INPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_2/h5ad_data_processed_4lockmasses_filtered_halfbrain_common/"
MSI_SAMPLE_FILES = [
    "halfbrain_yc_1_filtered_common.h5ad", "halfbrain_yc_2_filtered_common.h5ad",
    "halfbrain_yc_3_filtered_common.h5ad", "halfbrain_yc_4_filtered_common.h5ad",
    "halfbrain_yad_1_filtered_common.h5ad", "halfbrain_yad_2_filtered_common.h5ad",
    "halfbrain_yad_3_filtered_common.h5ad", "halfbrain_yad_4_filtered_common.h5ad",
    "halfbrain_ac_1_filtered_common.h5ad", "halfbrain_ac_2_filtered_common.h5ad",
    "halfbrain_ac_3_filtered_common.h5ad", "halfbrain_ac_4_filtered_common.h5ad",
    "halfbrain_aad_1_filtered_common.h5ad", "halfbrain_aad_2_filtered_common.h5ad",
    "halfbrain_aad_3_filtered_common.h5ad", "halfbrain_aad_4_filtered_common.h5ad"
]
MSI_SAMPLE_IDS = [
    "YC_1", "YC_2", "YC_3", "YC_4",
    "YAD_1", "YAD_2", "YAD_3", "YAD_4",
    "AC_1", "AC_2", "AC_3", "AC_4",
    "AAD_1", "AAD_2", "AAD_3", "AAD_4"
]

RNA_INPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_4/h5ad_data/genes_top_800/"
RNA_SAMPLE_FILES = [
    "YC_1.h5ad", "YC_2.h5ad", "YC_3.h5ad", "YC_4.h5ad",
    "YAD_1.h5ad", "YAD_2.h5ad", "YAD_3.h5ad", "YAD_4.h5ad",
    "AC_1.h5ad", "AC_2.h5ad", "AC_3.h5ad", "AC_4.h5ad",
    "AAD_1.h5ad", "AAD_2.h5ad", "AAD_3.h5ad", "AAD_4.h5ad"
]
RNA_SAMPLE_IDS = [
    "YC_1", "YC_2", "YC_3", "YC_4",
    "YAD_1", "YAD_2", "YAD_3", "YAD_4",
    "AC_1", "AC_2", "AC_3", "AC_4",
    "AAD_1", "AAD_2", "AAD_3", "AAD_4"
]

AAD_TARGET_GENES = [
    'Thy1', 'Mapt', 'Pmch', 'App', 'Apoe'
]


In [ ]:

# =============================================================================
# DATA CLASSES
# =============================================================================

@dataclass
class SpatialSignature:
    """Spatial signature for a gene or m/z feature"""
    sample_id: str
    feature_name: str
    feature_type: str  # 'gene' or 'mz'
    global_embedding: np.ndarray
    node_embeddings: np.ndarray
    node_importance: np.ndarray
    coordinates: np.ndarray
    raw_values: np.ndarray
    processed_values: np.ndarray  # After artifact removal
    spatial_coherence: float  # How spatially coherent the pattern is
    artifact_mask: np.ndarray  # True = artifact/outlier


In [ ]:

# =============================================================================
# ARTIFACT DETECTION AND FILTERING
# =============================================================================

class ArtifactDetector:
    """
    Detect and filter artifacts like blood spots in MSI data.
    
    Blood and other artifacts typically show:
    1. Extremely high intensity (outliers)
    2. Isolated spots (not spatially coherent)
    3. Present in few m/z channels with very high intensity
    """
    
    def __init__(
        self,
        zscore_threshold: float = 3.0,
        isolation_threshold: float = 0.3,
        min_region_size: int = 5
    ):
        self.zscore_threshold = zscore_threshold
        self.isolation_threshold = isolation_threshold
        self.min_region_size = min_region_size
    
    def detect_intensity_outliers(self, values: np.ndarray) -> np.ndarray:
        """Detect outliers based on intensity"""
        # Use robust z-score (based on median and MAD)
        median = np.median(values)
        mad = np.median(np.abs(values - median))
        if mad < 1e-8:
            mad = np.std(values)
        
        robust_zscore = 0.6745 * (values - median) / (mad + 1e-8)
        outliers = np.abs(robust_zscore) > self.zscore_threshold
        
        return outliers
    
    def detect_isolated_spots(
        self, 
        coords: np.ndarray, 
        values: np.ndarray,
        high_value_mask: np.ndarray,
        n_neighbors: int = 8
    ) -> np.ndarray:
        """
        Detect isolated bright spots (likely artifacts).
        True artifacts are bright spots surrounded by low values.
        """
        if high_value_mask.sum() == 0:
            return np.zeros(len(values), dtype=bool)
        
        nn = NearestNeighbors(n_neighbors=min(n_neighbors + 1, len(coords)))
        nn.fit(coords)
        _, indices = nn.kneighbors(coords)
        
        isolated = np.zeros(len(values), dtype=bool)
        
        # Check each high-value point
        high_indices = np.where(high_value_mask)[0]
        
        for idx in high_indices:
            neighbor_idx = indices[idx, 1:]  # Exclude self
            neighbor_values = values[neighbor_idx]
            center_value = values[idx]
            
            # If center is much higher than neighbors, it's isolated
            if center_value > 0:
                neighbor_ratio = neighbor_values.mean() / (center_value + 1e-8)
                if neighbor_ratio < self.isolation_threshold:
                    isolated[idx] = True
        
        return isolated
    
    def detect_artifacts(
        self,
        coords: np.ndarray,
        values: np.ndarray,
        data_type: str = 'msi'  # 'msi' or 'rna'
    ) -> Tuple[np.ndarray, dict]:
        """
        Comprehensive artifact detection.
        
        Returns:
            artifact_mask: Boolean array (True = artifact)
            stats: Dictionary with detection statistics
        """
        n_neighbors = 8 if data_type == 'msi' else 6
        
        # 1. Intensity outliers
        intensity_outliers = self.detect_intensity_outliers(values)
        
        # 2. Find high-value points (top 5%)
        threshold_95 = np.percentile(values, 95)
        high_value_mask = values > threshold_95
        
        # 3. Check if high values are isolated
        isolated = self.detect_isolated_spots(coords, values, high_value_mask, n_neighbors)
        
        # Combine: artifact = outlier AND isolated
        artifact_mask = intensity_outliers & isolated
        
        # Also mark extreme outliers (z > 5) regardless of isolation
        extreme_outliers = self.detect_intensity_outliers(values)
        median = np.median(values)
        mad = np.median(np.abs(values - median))
        if mad > 1e-8:
            extreme = 0.6745 * (values - median) / mad > 5.0
            artifact_mask = artifact_mask | extreme
        
        stats = {
            'n_intensity_outliers': intensity_outliers.sum(),
            'n_isolated': isolated.sum(),
            'n_artifacts': artifact_mask.sum(),
            'artifact_fraction': artifact_mask.mean()
        }
        
        return artifact_mask, stats
    
    def clean_values(
        self,
        coords: np.ndarray,
        values: np.ndarray,
        artifact_mask: np.ndarray,
        method: str = 'interpolate'
    ) -> np.ndarray:
        """
        Clean artifact values by interpolation or replacement.
        """
        cleaned = values.copy()
        
        if artifact_mask.sum() == 0:
            return cleaned
        
        if method == 'interpolate':
            # Interpolate artifact values from neighbors
            nn = NearestNeighbors(n_neighbors=10)
            nn.fit(coords[~artifact_mask])
            
            artifact_coords = coords[artifact_mask]
            _, indices = nn.kneighbors(artifact_coords)
            
            clean_values = values[~artifact_mask]
            interpolated = clean_values[indices].mean(axis=1)
            
            cleaned[artifact_mask] = interpolated
            
        elif method == 'median':
            # Replace with median
            cleaned[artifact_mask] = np.median(values[~artifact_mask])
        
        return cleaned



In [ ]:

# =============================================================================
# SPATIAL COHERENCE SCORING
# =============================================================================

def compute_spatial_coherence(
    coords: np.ndarray,
    values: np.ndarray,
    n_neighbors: int = 8
) -> float:
    """
    Compute how spatially coherent a pattern is.
    
    High coherence = smooth patterns with clear regions
    Low coherence = noisy, scattered bright spots
    
    Returns value in [0, 1] where 1 is highly coherent.
    """
    nn = NearestNeighbors(n_neighbors=n_neighbors + 1)
    nn.fit(coords)
    distances, indices = nn.kneighbors(coords)
    
    # Compute local similarity for each point
    local_correlations = []
    
    for i in range(len(coords)):
        center_val = values[i]
        neighbor_vals = values[indices[i, 1:]]
        
        # Local correlation
        if np.std(neighbor_vals) > 1e-8 and center_val != 0:
            local_corr = np.corrcoef([center_val] * len(neighbor_vals), neighbor_vals)[0, 1]
            if not np.isnan(local_corr):
                local_correlations.append(local_corr)
    
    if len(local_correlations) == 0:
        return 0.0
    
    # Average local correlation (transformed to [0, 1])
    mean_corr = np.mean(local_correlations)
    coherence = (mean_corr + 1) / 2  # Map [-1, 1] to [0, 1]
    
    return coherence


def compute_pattern_quality(
    coords: np.ndarray,
    values: np.ndarray,
    artifact_mask: np.ndarray,
    n_neighbors: int = 8
) -> dict:
    """
    Compute multiple quality metrics for a spatial pattern.
    """
    # Clean values for analysis
    clean_values = values.copy()
    clean_values[artifact_mask] = np.median(values[~artifact_mask])
    
    # Spatial coherence
    coherence = compute_spatial_coherence(coords, clean_values, n_neighbors)
    
    # Dynamic range (after artifact removal)
    clean_only = values[~artifact_mask]
    if len(clean_only) > 0:
        dynamic_range = (np.percentile(clean_only, 95) - np.percentile(clean_only, 5)) / (np.mean(clean_only) + 1e-8)
    else:
        dynamic_range = 0
    
    # Spatial variance ratio (explained by smooth pattern vs noise)
    # Compare variance of smoothed vs original
    from scipy.ndimage import gaussian_filter1d
    
    # Sort by spatial position and smooth
    sorted_idx = np.lexsort((coords[:, 1], coords[:, 0]))
    sorted_vals = clean_values[sorted_idx]
    smoothed = gaussian_filter1d(sorted_vals, sigma=3)
    
    total_var = np.var(sorted_vals)
    smooth_var = np.var(smoothed)
    noise_var = total_var - smooth_var
    
    signal_ratio = smooth_var / (total_var + 1e-8)
    
    return {
        'coherence': coherence,
        'dynamic_range': dynamic_range,
        'signal_ratio': signal_ratio,
        'quality_score': (coherence + signal_ratio) / 2
    }


In [ ]:


# =============================================================================
# PROPER SPATIAL GRAPH CONSTRUCTION
# =============================================================================

def build_hexagonal_graph(coords: np.ndarray, k: int = 6) -> torch.Tensor:
    """
    Build graph for hexagonal grid (Visium spatial transcriptomics).
    Each spot has 6 neighbors.
    """
    nn = NearestNeighbors(n_neighbors=k + 1)
    nn.fit(coords)
    distances, indices = nn.kneighbors(coords)
    
    # Filter to only include actual neighbors (within expected distance)
    median_dist = np.median(distances[:, 1])  # Distance to nearest neighbor
    max_dist = median_dist * 1.5  # Allow some tolerance
    
    edge_list = []
    for i in range(len(coords)):
        for j_idx in range(1, k + 1):
            j = indices[i, j_idx]
            if distances[i, j_idx] < max_dist:
                edge_list.append([i, j])
                edge_list.append([j, i])
    
    # Remove duplicates
    edge_set = set(map(tuple, edge_list))
    edge_list = list(edge_set)
    
    return torch.tensor(edge_list, dtype=torch.long).t().contiguous()


def build_cartesian_graph(coords: np.ndarray, k: int = 8) -> torch.Tensor:
    """
    Build graph for Cartesian grid (MSI data).
    Each pixel has 8 neighbors (including diagonals).
    """
    nn = NearestNeighbors(n_neighbors=k + 1)
    nn.fit(coords)
    distances, indices = nn.kneighbors(coords)
    
    # For Cartesian grid, neighbors can be at distance 1 (cardinal) or sqrt(2) (diagonal)
    median_dist = np.median(distances[:, 1])
    max_dist = median_dist * 1.5  # Should capture both cardinal and diagonal
    
    edge_list = []
    for i in range(len(coords)):
        for j_idx in range(1, k + 1):
            j = indices[i, j_idx]
            if distances[i, j_idx] < max_dist:
                edge_list.append([i, j])
                edge_list.append([j, i])
    
    edge_set = set(map(tuple, edge_list))
    edge_list = list(edge_set)
    
    return torch.tensor(edge_list, dtype=torch.long).t().contiguous()



In [ ]:

# =============================================================================
# IMPROVED SPATIAL GNN
# =============================================================================

class SpatialAwareGATConv(nn.Module):
    """
    GAT layer that's aware of spatial structure and penalizes attention to artifacts.
    """
    
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        heads: int = 4,
        concat: bool = True,
        dropout: float = 0.1
    ):
        super().__init__()
        
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.heads = heads
        self.concat = concat
        
        self.lin = nn.Linear(in_channels, heads * out_channels, bias=False)
        self.att_src = nn.Parameter(torch.Tensor(1, heads, out_channels))
        self.att_dst = nn.Parameter(torch.Tensor(1, heads, out_channels))
        
        # Spatial distance encoding
        self.dist_encoder = nn.Sequential(
            nn.Linear(1, heads),
            nn.Sigmoid()
        )
        
        if concat:
            self.bias = nn.Parameter(torch.Tensor(heads * out_channels))
        else:
            self.bias = nn.Parameter(torch.Tensor(out_channels))
        
        self.dropout = nn.Dropout(dropout)
        self.leaky_relu = nn.LeakyReLU(0.2)
        
        self.stored_attention = None
        self._reset_parameters()
    
    def _reset_parameters(self):
        nn.init.xavier_uniform_(self.lin.weight)
        nn.init.xavier_uniform_(self.att_src)
        nn.init.xavier_uniform_(self.att_dst)
        nn.init.zeros_(self.bias)
    
    def forward(
        self,
        x: torch.Tensor,
        edge_index: torch.Tensor,
        pos: torch.Tensor,
        artifact_mask: Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        """Forward with spatial awareness and artifact penalization"""
        N = x.size(0)
        
        # Add self-loops
        edge_index, _ = add_self_loops(edge_index, num_nodes=N)
        
        # Linear transformation
        x_transformed = self.lin(x).view(N, self.heads, self.out_channels)
        
        src, dst = edge_index[0], edge_index[1]
        
        # Attention scores
        alpha_src = (x_transformed * self.att_src).sum(dim=-1)  # [N, heads]
        alpha_dst = (x_transformed * self.att_dst).sum(dim=-1)  # [N, heads]
        
        alpha = alpha_src[src] + alpha_dst[dst]  # [E, heads]
        alpha = self.leaky_relu(alpha)
        
        # Spatial distance bias (closer neighbors get more attention)
        dist = torch.norm(pos[src] - pos[dst], dim=-1, keepdim=True)  # [E, 1]
        dist_normalized = dist / (dist.mean() + 1e-8)
        spatial_bias = self.dist_encoder(dist_normalized)  # [E, heads]
        
        # Reduce attention for distant nodes
        alpha = alpha - dist_normalized * 0.5
        
        # Penalize attention TO artifact nodes (but not FROM them)
        if artifact_mask is not None:
            artifact_penalty = artifact_mask[src].float() * (-5.0)  # Large negative = low attention
            alpha = alpha + artifact_penalty.unsqueeze(-1)
        
        # Softmax over neighbors
        alpha = softmax(alpha, dst, num_nodes=N)
        self.stored_attention = alpha.detach()
        
        alpha = self.dropout(alpha)
        
        # Aggregate
        out = torch.zeros(N, self.heads, self.out_channels, device=x.device)
        alpha_expanded = alpha.unsqueeze(-1)
        weighted = alpha_expanded * x_transformed[src]
        out.scatter_add_(0, dst.view(-1, 1, 1).expand(-1, self.heads, self.out_channels), weighted)
        
        if self.concat:
            out = out.view(N, self.heads * self.out_channels)
        else:
            out = out.mean(dim=1)
        
        return out + self.bias


class ImprovedSpatialGNN(nn.Module):
    """
    GNN that properly handles spatial structure and artifacts.
    """
    
    def __init__(
        self,
        input_dim: int = None,
        hidden_dim: int = 128,
        embedding_dim: int = 64,
        num_layers: int = 3,
        num_heads: int = 4,
        dropout: float = 0.1,
        projection_dim: int = 64
    ):
        super().__init__()
        
        self.hidden_dim = hidden_dim
        self.embedding_dim = embedding_dim
        self.projection_dim = projection_dim
        
        # Input projections
        self.input_projections = nn.ModuleDict()
        if input_dim is not None:
            self.input_projections[str(input_dim)] = nn.Sequential(
                nn.Linear(input_dim, projection_dim),
                nn.LayerNorm(projection_dim),
                nn.GELU()
            )
        
        # GAT layers
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        
        self.convs.append(SpatialAwareGATConv(
            projection_dim, hidden_dim, heads=num_heads, concat=True, dropout=dropout
        ))
        self.norms.append(nn.LayerNorm(hidden_dim * num_heads))
        
        for _ in range(num_layers - 2):
            self.convs.append(SpatialAwareGATConv(
                hidden_dim * num_heads, hidden_dim, heads=num_heads,
                concat=True, dropout=dropout
            ))
            self.norms.append(nn.LayerNorm(hidden_dim * num_heads))
        
        self.convs.append(SpatialAwareGATConv(
            hidden_dim * num_heads, embedding_dim, heads=1,
            concat=False, dropout=dropout
        ))
        self.norms.append(nn.LayerNorm(embedding_dim))
        
        # Importance network - predicts spatial importance
        self.importance_net = nn.Sequential(
            nn.Linear(embedding_dim + projection_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Linear(hidden_dim // 2, 1)
        )
        
        # Spatial coherence penalty network
        self.coherence_net = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Linear(hidden_dim // 2, 1),
            nn.Sigmoid()
        )
        
        # Global pooling
        self.pool_attention = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )
        
        # Reconstruction decoder
        self.decoder = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, projection_dim)
        )
        
        self.dropout = nn.Dropout(dropout)
    
    def _get_projection(self, input_dim: int) -> nn.Module:
        dim_key = str(input_dim)
        if dim_key not in self.input_projections:
            device = next(self.parameters()).device
            self.input_projections[dim_key] = nn.Sequential(
                nn.Linear(input_dim, self.projection_dim),
                nn.LayerNorm(self.projection_dim),
                nn.GELU()
            ).to(device)
        return self.input_projections[dim_key]
    
    def forward(
        self,
        x: torch.Tensor,
        edge_index: torch.Tensor,
        pos: torch.Tensor,
        artifact_mask: Optional[torch.Tensor] = None
    ) -> dict:
        """Forward pass with artifact awareness"""
        
        # Project input
        h_input = self._get_projection(x.size(1))(x)
        
        attention_weights = []
        h = h_input
        
        for i, (conv, norm) in enumerate(zip(self.convs, self.norms)):
            h_new = conv(h, edge_index, pos, artifact_mask)
            attention_weights.append(conv.stored_attention)
            h_new = norm(h_new)
            h = F.gelu(h_new) if i < len(self.convs) - 1 else h_new
            h = self.dropout(h)
        
        node_embeddings = h
        
        # Compute importance
        importance_input = torch.cat([node_embeddings, h_input], dim=-1)
        importance_logits = self.importance_net(importance_input).squeeze(-1)
        
        # Coherence score per node (how spatially coherent is this node's neighborhood)
        coherence_scores = self.coherence_net(node_embeddings).squeeze(-1)
        
        # Final importance = learned importance * coherence penalty
        # Low coherence = artifact-like = lower importance
        node_importance = torch.sigmoid(importance_logits) * coherence_scores
        
        # Penalize artifact nodes
        if artifact_mask is not None:
            node_importance = node_importance * (~artifact_mask).float()
        
        # Normalize importance
        node_importance = node_importance / (node_importance.max() + 1e-8)
        
        # Weighted embeddings
        weight = node_importance / (node_importance.sum() + 1e-8) * x.size(0)
        weighted_embeddings = node_embeddings * weight.unsqueeze(-1)
        
        # Global embedding
        attn_scores = self.pool_attention(weighted_embeddings).squeeze(-1)
        attn_weights = F.softmax(attn_scores, dim=0)
        global_embedding = (attn_weights.unsqueeze(-1) * weighted_embeddings).sum(dim=0)
        
        # Reconstruction
        reconstructed = self.decoder(node_embeddings)
        
        return {
            'node_embeddings': node_embeddings,
            'weighted_embeddings': weighted_embeddings,
            'global_embedding': global_embedding,
            'node_importance': node_importance,
            'coherence_scores': coherence_scores,
            'attention_weights': attention_weights,
            'reconstructed': reconstructed
        }


In [ ]:

# =============================================================================
# IMPROVED LOSS FUNCTION
# =============================================================================

class SpatialCoherenceLoss(nn.Module):
    """Loss function that encourages spatially coherent importance maps"""
    
    def __init__(
        self,
        reconstruction_weight: float = 1.0,
        smoothness_weight: float = 0.5,
        coherence_weight: float = 0.3,
        contrastive_weight: float = 0.3
    ):
        super().__init__()
        self.reconstruction_weight = reconstruction_weight
        self.smoothness_weight = smoothness_weight
        self.coherence_weight = coherence_weight
        self.contrastive_weight = contrastive_weight
    
    def forward(
        self,
        output: dict,
        input_features: torch.Tensor,
        pos: torch.Tensor,
        edge_index: torch.Tensor,
        artifact_mask: Optional[torch.Tensor] = None
    ) -> Tuple[torch.Tensor, dict]:
        
        losses = {}
        
        # 1. Reconstruction
        recon_loss = F.mse_loss(output['reconstructed'], input_features)
        losses['reconstruction'] = recon_loss * self.reconstruction_weight
        
        # 2. Spatial smoothness of embeddings
        src, dst = edge_index[0], edge_index[1]
        emb_diff = output['node_embeddings'][src] - output['node_embeddings'][dst]
        smooth_loss = (emb_diff ** 2).mean()
        losses['smoothness'] = smooth_loss * self.smoothness_weight / (output['node_embeddings'].size(-1))
        
        # 3. Importance coherence - importance should be spatially smooth
        imp = output['node_importance']
        imp_diff = (imp[src] - imp[dst]) ** 2
        imp_smooth_loss = imp_diff.mean()
        losses['importance_coherence'] = imp_smooth_loss * self.coherence_weight
        
        # 4. Contrastive - different regions should have different embeddings
        N = pos.size(0)
        n_samples = min(500, N)
        idx1 = torch.randint(0, N, (n_samples,), device=pos.device)
        idx2 = torch.randint(0, N, (n_samples,), device=pos.device)
        
        spatial_dist = torch.norm(pos[idx1] - pos[idx2], dim=-1)
        emb_sim = F.cosine_similarity(
            output['node_embeddings'][idx1],
            output['node_embeddings'][idx2]
        )
        
        # Far nodes should be dissimilar
        far_mask = spatial_dist > spatial_dist.median()
        if far_mask.sum() > 0:
            contrastive_loss = emb_sim[far_mask].mean()  # Should be low
            losses['contrastive'] = contrastive_loss * self.contrastive_weight
        else:
            losses['contrastive'] = torch.tensor(0.0, device=pos.device)
        
        total = sum(losses.values())
        loss_dict = {k: v.item() for k, v in losses.items()}
        loss_dict['total'] = total.item()
        
        return total, loss_dict



In [ ]:

# =============================================================================
# MAIN MATCHER CLASS
# =============================================================================

class ImprovedMatcher:
    """
    Improved gene-to-m/z matcher with artifact handling and proper spatial structure.
    """
    
    def __init__(
        self,
        output_dir: str = './gene_to_mz_results_v3',
        device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
    ):
        self.device = device
        self.output_dir = output_dir
        
        for subdir in ['saliency_maps', 'gene_visualizations', 'training_curves',
                       'artifact_analysis', 'embeddings']:
            os.makedirs(os.path.join(output_dir, subdir), exist_ok=True)
        
        self.rna_data = {}
        self.msi_data = {}
        self.rna_model = None
        self.msi_model = None
        
        self.artifact_detector = ArtifactDetector()
        
        self.gene_signatures = defaultdict(dict)
        self.mz_signatures = defaultdict(dict)
        
        # Track m/z quality to filter out artifact-dominated features
        self.mz_quality = {}
    
    def load_all_data(self):
        """Load all data"""
        print("Loading RNA-seq data...")
        for file, sample_id in zip(RNA_SAMPLE_FILES, RNA_SAMPLE_IDS):
            path = os.path.join(RNA_INPUT_FOLDER, file)
            if os.path.exists(path):
                self.rna_data[sample_id] = sc.read_h5ad(path)
                print(f"  {sample_id}: {self.rna_data[sample_id].shape}")
        
        print("\nLoading MSI data...")
        for file, sample_id in zip(MSI_SAMPLE_FILES, MSI_SAMPLE_IDS):
            path = os.path.join(MSI_INPUT_FOLDER, file)
            if os.path.exists(path):
                self.msi_data[sample_id] = sc.read_h5ad(path)
                print(f"  {sample_id}: {self.msi_data[sample_id].shape}")
    
    def precompute_mz_quality(self):
        """
        Precompute quality scores for all m/z features to filter artifacts.
        """
        print("\nPrecomputing m/z quality scores (filtering artifact features)...")
        
        for sample_id, adata in self.msi_data.items():
            print(f"  {sample_id}...")
            coords = np.column_stack([adata.obs['x_um'].values, adata.obs['y_um'].values])
            
            if hasattr(adata.X, 'toarray'):
                data = adata.X.toarray()
            else:
                data = adata.X.copy()
            
            mz_names = list(adata.var_names)
            
            quality_scores = {}
            artifact_features = []
            
            for i, mz_name in enumerate(mz_names):
                values = data[:, i]
                
                # Detect artifacts
                artifact_mask, stats = self.artifact_detector.detect_artifacts(
                    coords, values, data_type='msi'
                )
                
                # Compute quality
                quality = compute_pattern_quality(coords, values, artifact_mask, n_neighbors=8)
                
                quality_scores[mz_name] = {
                    'coherence': quality['coherence'],
                    'quality_score': quality['quality_score'],
                    'artifact_fraction': stats['artifact_fraction']
                }
                
                # Flag high-artifact features
                if stats['artifact_fraction'] > 0.05 or quality['coherence'] < 0.3:
                    artifact_features.append(mz_name)
            
            self.mz_quality[sample_id] = {
                'scores': quality_scores,
                'artifact_features': set(artifact_features)
            }
            
            print(f"    {len(artifact_features)}/{len(mz_names)} features flagged as artifact-dominated")
    
    def check_gene_availability(self) -> Dict[str, Dict[str, bool]]:
        """Check gene availability"""
        gene_availability = {}
        for gene in AAD_TARGET_GENES:
            gene_availability[gene] = {}
            for sample_id in RNA_SAMPLE_IDS:
                if sample_id in self.rna_data:
                    gene_availability[gene][sample_id] = gene in self.rna_data[sample_id].var_names
                else:
                    gene_availability[gene][sample_id] = False
        
        print("\nGene availability:")
        for gene in AAD_TARGET_GENES:
            available = sum(gene_availability[gene].values())
            print(f"  {gene}: {available}/{len(RNA_SAMPLE_IDS)} samples")
        
        return gene_availability
    
    def prepare_rna_data(
        self,
        coords: np.ndarray,
        features: np.ndarray
    ) -> Data:
        """Prepare RNA data with hexagonal graph (6 neighbors)"""
        if features.ndim == 1:
            features = features.reshape(-1, 1)
        
        # Robust scaling
        scaler = RobustScaler()
        features_scaled = scaler.fit_transform(features)
        
        # Build hexagonal graph
        edge_index = build_hexagonal_graph(coords, k=6)
        
        pos = (coords - coords.mean(axis=0)) / (coords.std(axis=0) + 1e-8)
        
        return Data(
            x=torch.tensor(features_scaled, dtype=torch.float32),
            edge_index=edge_index,
            pos=torch.tensor(pos, dtype=torch.float32)
        )
    
    def prepare_msi_data(
        self,
        coords: np.ndarray,
        features: np.ndarray
    ) -> Tuple[Data, np.ndarray]:
        """Prepare MSI data with Cartesian graph (8 neighbors) and artifact detection"""
        if features.ndim == 1:
            features = features.reshape(-1, 1)
        
        # Detect artifacts
        artifact_mask, _ = self.artifact_detector.detect_artifacts(
            coords, features.mean(axis=1) if features.ndim > 1 else features,
            data_type='msi'
        )
        
        # Clean features
        features_clean = features.copy()
        for i in range(features.shape[1]):
            col_artifacts, _ = self.artifact_detector.detect_artifacts(
                coords, features[:, i], data_type='msi'
            )
            features_clean[:, i] = self.artifact_detector.clean_values(
                coords, features[:, i], col_artifacts
            )
        
        # Robust scaling
        scaler = RobustScaler()
        features_scaled = scaler.fit_transform(features_clean)
        
        # Build Cartesian graph
        edge_index = build_cartesian_graph(coords, k=8)
        
        pos = (coords - coords.mean(axis=0)) / (coords.std(axis=0) + 1e-8)
        
        data = Data(
            x=torch.tensor(features_scaled, dtype=torch.float32),
            edge_index=edge_index,
            pos=torch.tensor(pos, dtype=torch.float32)
        )
        
        return data, artifact_mask
    
    def train_model(
        self,
        data_dict: Dict[str, Data],
        artifact_masks: Dict[str, np.ndarray],
        model_type: str,
        epochs: int = 200,
        lr: float = 0.001
    ) -> ImprovedSpatialGNN:
        """Train model"""
        print(f"\nTraining {model_type.upper()} model...")
        
        first_data = list(data_dict.values())[0]
        input_dim = first_data.x.size(1)
        
        model = ImprovedSpatialGNN(
            input_dim=input_dim,
            hidden_dim=128,
            embedding_dim=64,
            num_layers=3,
            num_heads=4,
            projection_dim=64
        ).to(self.device)
        
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        
        warmup_epochs = 20
        def lr_lambda(epoch):
            if epoch < warmup_epochs:
                return epoch / warmup_epochs
            return 0.5 * (1 + np.cos(np.pi * (epoch - warmup_epochs) / (epochs - warmup_epochs)))
        
        scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
        loss_fn = SpatialCoherenceLoss()
        
        model.train()
        loss_history = []
        
        for epoch in range(epochs):
            total_loss = 0
            
            for sample_id, data in data_dict.items():
                data = data.to(self.device)
                
                artifact_mask = artifact_masks.get(sample_id)
                if artifact_mask is not None:
                    artifact_mask = torch.tensor(artifact_mask, dtype=torch.bool, device=self.device)
                
                optimizer.zero_grad()
                
                input_features = model._get_projection(data.x.size(1))(data.x)
                output = model(data.x, data.edge_index, data.pos, artifact_mask)
                
                loss, _ = loss_fn(output, input_features, data.pos, data.edge_index, artifact_mask)
                
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                
                total_loss += loss.item()
            
            scheduler.step()
            avg_loss = total_loss / len(data_dict)
            loss_history.append(avg_loss)
            
            if (epoch + 1) % 50 == 0:
                print(f"  Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")
        
        print(f"  Final loss: {loss_history[-1]:.4f}")
        
        # Plot loss
        plt.figure(figsize=(10, 5))
        plt.plot(loss_history)
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.title(f'{model_type.upper()} Training Loss')
        plt.axhline(y=1.0, color='r', linestyle='--', alpha=0.5)
        plt.grid(True, alpha=0.3)
        plt.savefig(os.path.join(self.output_dir, 'training_curves', f'{model_type}_loss.png'), dpi=150)
        plt.close()
        
        return model
    
    def extract_gene_signature(
        self,
        sample_id: str,
        gene: str,
        model: ImprovedSpatialGNN
    ) -> Optional[SpatialSignature]:
        """Extract gene signature"""
        if sample_id not in self.rna_data:
            return None
        
        adata = self.rna_data[sample_id]
        if gene not in adata.var_names:
            return None
        
        coords = np.column_stack([adata.obs['x_um'].values, adata.obs['y_um'].values])
        
        gene_idx = list(adata.var_names).index(gene)
        if hasattr(adata.X, 'toarray'):
            gene_expr = adata.X[:, gene_idx].toarray().flatten()
        else:
            gene_expr = adata.X[:, gene_idx].flatten()
        
        # Detect artifacts (less common in RNA but possible)
        artifact_mask, _ = self.artifact_detector.detect_artifacts(coords, gene_expr, 'rna')
        cleaned = self.artifact_detector.clean_values(coords, gene_expr, artifact_mask)
        
        data = self.prepare_rna_data(coords, cleaned)
        data = data.to(self.device)
        
        model.eval()
        with torch.no_grad():
            output = model(data.x, data.edge_index, data.pos, None)
        
        # Compute spatial coherence
        coherence = compute_spatial_coherence(coords, cleaned, n_neighbors=6)
        
        sig = SpatialSignature(
            sample_id=sample_id,
            feature_name=gene,
            feature_type='gene',
            global_embedding=output['global_embedding'].cpu().numpy(),
            node_embeddings=output['weighted_embeddings'].cpu().numpy(),
            node_importance=output['node_importance'].cpu().numpy(),
            coordinates=coords,
            raw_values=gene_expr,
            processed_values=cleaned,
            spatial_coherence=coherence,
            artifact_mask=artifact_mask
        )
        
        self.gene_signatures[gene][sample_id] = sig
        return sig
    
    def extract_mz_signature(
        self,
        sample_id: str,
        mz_name: str,
        mz_values: np.ndarray,
        coords: np.ndarray,
        model: ImprovedSpatialGNN
    ) -> Optional[SpatialSignature]:
        """Extract single m/z signature"""
        
        # Detect artifacts
        artifact_mask, stats = self.artifact_detector.detect_artifacts(coords, mz_values, 'msi')
        cleaned = self.artifact_detector.clean_values(coords, mz_values, artifact_mask)
        
        # Compute quality
        quality = compute_pattern_quality(coords, mz_values, artifact_mask, n_neighbors=8)
        
        # Skip if too many artifacts or low coherence
        if stats['artifact_fraction'] > 0.1 or quality['coherence'] < 0.2:
            return None
        
        data, _ = self.prepare_msi_data(coords, cleaned)
        data = data.to(self.device)
        artifact_tensor = torch.tensor(artifact_mask, dtype=torch.bool, device=self.device)
        
        model.eval()
        with torch.no_grad():
            output = model(data.x, data.edge_index, data.pos, artifact_tensor)
        
        sig = SpatialSignature(
            sample_id=sample_id,
            feature_name=mz_name,
            feature_type='mz',
            global_embedding=output['global_embedding'].cpu().numpy(),
            node_embeddings=output['weighted_embeddings'].cpu().numpy(),
            node_importance=output['node_importance'].cpu().numpy(),
            coordinates=coords,
            raw_values=mz_values,
            processed_values=cleaned,
            spatial_coherence=quality['coherence'],
            artifact_mask=artifact_mask
        )
        
        return sig
    
    def compute_similarity(
        self,
        gene_sig: SpatialSignature,
        mz_sig: SpatialSignature
    ) -> dict:
        """Compute similarity between gene and m/z signatures"""
        
        # Embedding similarity
        g_emb = gene_sig.global_embedding.flatten()
        m_emb = mz_sig.global_embedding.flatten()
        
        cosine = np.dot(g_emb, m_emb) / (np.linalg.norm(g_emb) * np.linalg.norm(m_emb) + 1e-8)
        euclidean = 1 / (1 + np.linalg.norm(g_emb - m_emb))
        
        # Importance pattern overlap (interpolated to common grid)
        imp_overlap = self._compute_importance_overlap(gene_sig, mz_sig)
        
        # Raw correlation (using cleaned values)
        raw_pearson = self._compute_raw_correlation(gene_sig, mz_sig)
        
        # Quality bonus - prefer high coherence matches
        quality_bonus = (gene_sig.spatial_coherence + mz_sig.spatial_coherence) / 2
        
        return {
            'embedding_cosine': cosine,
            'embedding_euclidean': euclidean,
            'importance_overlap': imp_overlap,
            'raw_pearson': raw_pearson,
            'quality_bonus': quality_bonus,
            'gene_coherence': gene_sig.spatial_coherence,
            'mz_coherence': mz_sig.spatial_coherence
        }
    
    def _compute_importance_overlap(
        self,
        sig1: SpatialSignature,
        sig2: SpatialSignature,
        grid_res: int = 50
    ) -> float:
        """Compute importance map overlap"""
        x_min = min(sig1.coordinates[:, 0].min(), sig2.coordinates[:, 0].min())
        x_max = max(sig1.coordinates[:, 0].max(), sig2.coordinates[:, 0].max())
        y_min = min(sig1.coordinates[:, 1].min(), sig2.coordinates[:, 1].min())
        y_max = max(sig1.coordinates[:, 1].max(), sig2.coordinates[:, 1].max())
        
        grid_x, grid_y = np.meshgrid(
            np.linspace(x_min, x_max, grid_res),
            np.linspace(y_min, y_max, grid_res)
        )
        
        imp1 = griddata(sig1.coordinates, sig1.node_importance, (grid_x, grid_y),
                       method='linear', fill_value=0)
        imp2 = griddata(sig2.coordinates, sig2.node_importance, (grid_x, grid_y),
                       method='linear', fill_value=0)
        
        imp1 = imp1 / (imp1.max() + 1e-8)
        imp2 = imp2 / (imp2.max() + 1e-8)
        
        intersection = np.minimum(imp1, imp2).sum()
        union = np.maximum(imp1, imp2).sum()
        
        return intersection / (union + 1e-8)
    
    def _compute_raw_correlation(
        self,
        sig1: SpatialSignature,
        sig2: SpatialSignature,
        grid_res: int = 50
    ) -> float:
        """Compute raw value correlation"""
        x_min = min(sig1.coordinates[:, 0].min(), sig2.coordinates[:, 0].min())
        x_max = max(sig1.coordinates[:, 0].max(), sig2.coordinates[:, 0].max())
        y_min = min(sig1.coordinates[:, 1].min(), sig2.coordinates[:, 1].min())
        y_max = max(sig1.coordinates[:, 1].max(), sig2.coordinates[:, 1].max())
        
        grid_x, grid_y = np.meshgrid(
            np.linspace(x_min, x_max, grid_res),
            np.linspace(y_min, y_max, grid_res)
        )
        
        v1 = griddata(sig1.coordinates, sig1.processed_values, (grid_x, grid_y),
                     method='linear', fill_value=np.nan)
        v2 = griddata(sig2.coordinates, sig2.processed_values, (grid_x, grid_y),
                     method='linear', fill_value=np.nan)
        
        mask = ~(np.isnan(v1) | np.isnan(v2))
        if mask.sum() < 10:
            return 0.0
        
        r, _ = pearsonr(v1[mask], v2[mask])
        return r if not np.isnan(r) else 0.0
    
    def find_matches(
        self,
        gene_sig: SpatialSignature,
        mz_signatures: Dict[str, SpatialSignature],
        top_k: int = 20
    ) -> pd.DataFrame:
        """Find best m/z matches for a gene"""
        matches = []
        
        for mz_name, mz_sig in mz_signatures.items():
            sim = self.compute_similarity(gene_sig, mz_sig)
            
            # Combined score with quality weighting
            combined = (
                0.35 * sim['embedding_cosine'] +
                0.25 * sim['importance_overlap'] +
                0.20 * max(sim['raw_pearson'], 0) +
                0.10 * sim['embedding_euclidean'] +
                0.10 * sim['quality_bonus']
            )
            
            matches.append({
                'gene': gene_sig.feature_name,
                'rna_sample': gene_sig.sample_id,
                'mz_feature': mz_name,
                'msi_sample': mz_sig.sample_id,
                **sim,
                'combined_score': combined
            })
        
        df = pd.DataFrame(matches)
        return df.sort_values('combined_score', ascending=False).head(top_k)
    
    def visualize_saliency(
        self,
        sig: SpatialSignature,
        save_path: str
    ):
        """Visualize saliency with artifact highlighting"""
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        
        # 1. Raw values
        im1 = axes[0, 0].scatter(
            sig.coordinates[:, 0], sig.coordinates[:, 1],
            c=sig.raw_values, cmap='viridis', s=10, alpha=0.8
        )
        axes[0, 0].set_title(f'Raw: {sig.feature_name}', fontweight='bold')
        plt.colorbar(im1, ax=axes[0, 0])
        
        # 2. Artifact mask
        axes[0, 1].scatter(
            sig.coordinates[~sig.artifact_mask, 0],
            sig.coordinates[~sig.artifact_mask, 1],
            c='blue', s=5, alpha=0.5, label='Clean'
        )
        axes[0, 1].scatter(
            sig.coordinates[sig.artifact_mask, 0],
            sig.coordinates[sig.artifact_mask, 1],
            c='red', s=20, alpha=0.9, label='Artifact'
        )
        axes[0, 1].set_title(f'Artifacts ({sig.artifact_mask.sum()} points)', fontweight='bold')
        axes[0, 1].legend()
        
        # 3. Cleaned values
        im3 = axes[0, 2].scatter(
            sig.coordinates[:, 0], sig.coordinates[:, 1],
            c=sig.processed_values, cmap='viridis', s=10, alpha=0.8
        )
        axes[0, 2].set_title('Cleaned Values', fontweight='bold')
        plt.colorbar(im3, ax=axes[0, 2])
        
        # 4. Importance map
        im4 = axes[1, 0].scatter(
            sig.coordinates[:, 0], sig.coordinates[:, 1],
            c=sig.node_importance, cmap='hot', s=10, alpha=0.8
        )
        axes[1, 0].set_title(f'Importance (Coherence: {sig.spatial_coherence:.2f})', fontweight='bold')
        plt.colorbar(im4, ax=axes[1, 0])
        
        # 5. Importance-weighted values
        weighted = sig.processed_values * sig.node_importance
        weighted = (weighted - weighted.min()) / (weighted.max() - weighted.min() + 1e-8)
        im5 = axes[1, 1].scatter(
            sig.coordinates[:, 0], sig.coordinates[:, 1],
            c=weighted, cmap='plasma', s=10, alpha=0.8
        )
        axes[1, 1].set_title('Importance-Weighted', fontweight='bold')
        plt.colorbar(im5, ax=axes[1, 1])
        
        # 6. Top important regions
        threshold = np.percentile(sig.node_importance, 80)
        important = sig.node_importance >= threshold
        
        axes[1, 2].scatter(
            sig.coordinates[:, 0], sig.coordinates[:, 1],
            c='lightgray', s=5, alpha=0.3
        )
        axes[1, 2].scatter(
            sig.coordinates[important, 0],
            sig.coordinates[important, 1],
            c=sig.processed_values[important], cmap='Reds', s=15, alpha=0.9
        )
        axes[1, 2].set_title('Top 20% Important Regions', fontweight='bold')
        
        plt.suptitle(f'{sig.feature_type.upper()}: {sig.feature_name} ({sig.sample_id})',
                    fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(save_path, dpi=200, bbox_inches='tight')
        plt.close()
    
    def run_analysis(
        self,
        epochs: int = 200,
        top_k: int = 20,
        samples_per_group: int = 2
    ) -> pd.DataFrame:
        """Run full analysis"""
        print("\n" + "="*70)
        print("GENE-TO-M/Z MATCHING V3 - WITH ARTIFACT FILTERING")
        print("="*70)
        
        gene_availability = self.check_gene_availability()
        
        # Precompute m/z quality
        self.precompute_mz_quality()
        
        # ===== TRAINING =====
        print("\n" + "-"*70)
        print("PHASE 1: Training Models")
        print("-"*70)
        
        # Prepare RNA training data
        print("\nPreparing RNA data (hexagonal, 6 neighbors)...")
        rna_train = {}
        rna_artifacts = {}
        
        for sample_id in list(self.rna_data.keys())[:4]:
            adata = self.rna_data[sample_id]
            coords = np.column_stack([adata.obs['x_um'].values, adata.obs['y_um'].values])
            
            if hasattr(adata.X, 'toarray'):
                features = adata.X[:, :50].toarray()
            else:
                features = adata.X[:, :50].copy()
            
            rna_train[sample_id] = self.prepare_rna_data(coords, features)
            rna_artifacts[sample_id] = np.zeros(len(coords), dtype=bool)
            print(f"  {sample_id}: {rna_train[sample_id].x.shape}")
        
        self.rna_model = self.train_model(rna_train, rna_artifacts, 'rna', epochs)
        
        # Prepare MSI training data
        print("\nPreparing MSI data (Cartesian, 8 neighbors)...")
        msi_train = {}
        msi_artifacts = {}
        
        for sample_id in list(self.msi_data.keys())[:4]:
            adata = self.msi_data[sample_id]
            coords = np.column_stack([adata.obs['x_um'].values, adata.obs['y_um'].values])
            
            if hasattr(adata.X, 'toarray'):
                features = adata.X[:, :50].toarray()
            else:
                features = adata.X[:, :50].copy()
            
            data, artifacts = self.prepare_msi_data(coords, features)
            msi_train[sample_id] = data
            msi_artifacts[sample_id] = artifacts
            print(f"  {sample_id}: {data.x.shape}, artifacts: {artifacts.sum()}")
        
        self.msi_model = self.train_model(msi_train, msi_artifacts, 'msi', epochs)
        
        # ===== EXTRACTION AND MATCHING =====
        print("\n" + "-"*70)
        print("PHASE 2: Extracting Signatures and Matching")
        print("-"*70)
        
        all_results = []
        
        # Process all groups
        groups = ['YC', 'YAD', 'AC', 'AAD']
        
        for gene in AAD_TARGET_GENES:
            print(f"\n{'='*50}")
            print(f"Gene: {gene}")
            print(f"{'='*50}")
            
            available = [s for s, a in gene_availability[gene].items() if a]
            if not available:
                print("  Not available")
                continue
            
            # Sample from each group
            samples_to_process = []
            for group in groups:
                group_samples = [s for s in available if s.startswith(group)]
                samples_to_process.extend(group_samples[:samples_per_group])
            
            for rna_sample in samples_to_process:
                print(f"\n  RNA Sample: {rna_sample}")
                
                gene_sig = self.extract_gene_signature(rna_sample, gene, self.rna_model)
                if gene_sig is None:
                    continue
                
                # Save saliency map
                self.visualize_saliency(
                    gene_sig,
                    os.path.join(self.output_dir, 'saliency_maps', f'{gene}_{rna_sample}.png')
                )
                
                # Match against corresponding MSI sample (same animal)
                msi_sample = rna_sample  # Same sample ID
                if msi_sample not in self.msi_data:
                    continue
                
                print(f"    Matching vs MSI {msi_sample}...")
                
                adata = self.msi_data[msi_sample]
                coords = np.column_stack([adata.obs['x_um'].values, adata.obs['y_um'].values])
                
                if hasattr(adata.X, 'toarray'):
                    msi_data = adata.X.toarray()
                else:
                    msi_data = adata.X.copy()
                
                mz_names = list(adata.var_names)
                
                # Filter out artifact-dominated m/z features
                artifact_mz = self.mz_quality.get(msi_sample, {}).get('artifact_features', set())
                clean_mz_names = [m for m in mz_names if m not in artifact_mz]
                
                print(f"    Processing {len(clean_mz_names)}/{len(mz_names)} clean m/z features...")
                
                mz_sigs = {}
                for mz_name in clean_mz_names:
                    mz_idx = mz_names.index(mz_name)
                    mz_values = msi_data[:, mz_idx]
                    
                    sig = self.extract_mz_signature(
                        msi_sample, mz_name, mz_values, coords, self.msi_model
                    )
                    if sig is not None:
                        mz_sigs[mz_name] = sig
                
                if mz_sigs:
                    matches = self.find_matches(gene_sig, mz_sigs, top_k)
                    all_results.append(matches)
                    
                    if len(matches) > 0:
                        top = matches.iloc[0]
                        print(f"      Top match: m/z {top['mz_feature']}")
                        print(f"        Combined: {top['combined_score']:.3f}")
                        print(f"        Emb cosine: {top['embedding_cosine']:.3f}")
                        print(f"        Imp overlap: {top['importance_overlap']:.3f}")
                        print(f"        Raw Pearson: {top['raw_pearson']:.3f}")
                        print(f"        m/z coherence: {top['mz_coherence']:.3f}")
        
        # Combine results
        if all_results:
            results = pd.concat(all_results, ignore_index=True)
            results = results.sort_values('combined_score', ascending=False)
            
            output_file = os.path.join(self.output_dir, 'gene_to_mz_matches_v3.csv')
            results.to_csv(output_file, index=False)
            print(f"\nResults saved to: {output_file}")
            
            # Print summary
            print("\n" + "="*70)
            print("TOP MATCHES PER GENE")
            print("="*70)
            
            for gene in AAD_TARGET_GENES:
                gene_results = results[results['gene'] == gene]
                if len(gene_results) > 0:
                    top = gene_results.iloc[0]
                    print(f"\n{gene}:")
                    print(f"  Best m/z: {top['mz_feature']}")
                    print(f"  Score: {top['combined_score']:.3f}")
                    print(f"  Coherence: {top['mz_coherence']:.3f}")
            
            return results
        
        return None



In [ ]:


def main():
    print("="*70)
    print("GENE-TO-M/Z MATCHING V3")
    print("With Artifact Filtering & Proper Spatial Structure")
    print("="*70)
    
    matcher = ImprovedMatcher(output_dir='./gene_to_mz_results_v3')
    matcher.load_all_data()
    
    results = matcher.run_analysis(epochs=200, top_k=20, samples_per_group=2)
    
    print("\n" + "="*70)
    print("COMPLETE!")
    print("="*70)
    
    return matcher, results


In [ ]:

if __name__ == "__main__":
    matcher, results = main()